In [1]:
import random
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Optional QAT import
try:
    from torchao.quantization.prototype.qat import Int8DynActInt4WeightQATQuantizer
    HAS_TORCHAO = True
except ImportError:
    HAS_TORCHAO = False


In [2]:
# =========================================================
# 1. Config
# =========================================================
@dataclass
class Config:
    # data
    vocab_size: int = 5000
    max_text_len: int = 32
    num_image_tokens: int = 8
    image_feat_dim: int = 128
    behavior_dim: int = 16

    # dataset size
    train_size: int = 1000
    test_size: int = 200

    # model
    text_embed_dim: int = 128
    hidden_dim: int = 128
    num_heads: int = 4
    num_layers: int = 2
    ff_dim: int = 256
    dropout: float = 0.1

    # train
    batch_size: int = 32
    epochs: int = 5
    lr: float = 1e-3
    weight_decay: float = 1e-4

    # multitask loss
    alpha: float = 1.0   # harmful classification weight
    beta: float = 0.5    # report regression weight
    c: float = 5.0       # view-weight cap

    # qat
    use_qat: bool = True
    qat_finetune_epochs: int = 2
    seed: int = 42

In [3]:
# =========================================================
# 2. Reproducibility
# =========================================================
def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

In [4]:
# =========================================================
# 3. Dataset
# =========================================================
class MultimodalModerationDataset(Dataset):
    """
    Each sample:
      - text_ids: variable-length token ids
      - image_feats: [num_image_tokens, image_feat_dim]
      - behavior_feats: [behavior_dim]
      - harmful: 0/1
      - report_rate: float
      - views: int/float
    """

    def __init__(
        self,
        num_samples: int,
        vocab_size: int,
        max_text_len: int,
        num_image_tokens: int,
        image_feat_dim: int,
        behavior_dim: int,
    ):
        self.samples = []

        for _ in range(num_samples):
            text_len = random.randint(8, max_text_len)
            text_ids = torch.randint(1, vocab_size, (text_len,))  # 0 reserved for padding

            image_feats = torch.randn(num_image_tokens, image_feat_dim)
            behavior_feats = torch.randn(behavior_dim)

            harmful = torch.randint(0, 2, (1,)).item()
            report_rate = torch.rand(1).item()
            views = random.randint(1, 10000)

            self.samples.append(
                {
                    "text_ids": text_ids,
                    "image_feats": image_feats,
                    "behavior_feats": behavior_feats,
                    "harmful": harmful,
                    "report_rate": report_rate,
                    "views": views,
                }
            )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

In [5]:
def collate_fn(batch):
    batch_size = len(batch)
    max_len = max(len(x["text_ids"]) for x in batch)

    text_ids = torch.zeros(batch_size, max_len, dtype=torch.long)
    text_mask = torch.zeros(batch_size, max_len, dtype=torch.long)

    image_feats = []
    behavior_feats = []
    harmful = []
    report_rate = []
    views = []

    for i, sample in enumerate(batch):
        seq = sample["text_ids"]
        text_ids[i, : len(seq)] = seq
        text_mask[i, : len(seq)] = 1

        image_feats.append(sample["image_feats"])
        behavior_feats.append(sample["behavior_feats"])
        harmful.append(sample["harmful"])
        report_rate.append(sample["report_rate"])
        views.append(sample["views"])

    return {
        "text_ids": text_ids,
        "text_mask": text_mask,
        "image_feats": torch.stack(image_feats),          # [B, I, D_img]
        "behavior_feats": torch.stack(behavior_feats),    # [B, D_beh]
        "harmful": torch.tensor(harmful, dtype=torch.float32),
        "report_rate": torch.tensor(report_rate, dtype=torch.float32),
        "views": torch.tensor(views, dtype=torch.float32),
    }


In [6]:
# =========================================================
# 4. Model
# =========================================================
class MultimodalClassifier(nn.Module):
    """
    Architecture:
      text -> embedding + projection
      image feats -> projection
      text/image tokens -> TransformerEncoder
      behavior features -> MLP
      multimodal repr + behavior repr -> fusion MLP
      -> harmful/benign head + report-rate head
    """

    def __init__(
        self,
        vocab_size: int,
        text_embed_dim: int,
        image_feat_dim: int,
        behavior_dim: int,
        hidden_dim: int,
        num_heads: int,
        num_layers: int,
        ff_dim: int,
        max_text_len: int,
        dropout: float,
    ):
        super().__init__()

        # text branch
        self.text_embedding = nn.Embedding(vocab_size, text_embed_dim, padding_idx=0)
        self.text_proj = nn.Linear(text_embed_dim, hidden_dim)
        self.text_pos_embedding = nn.Parameter(torch.randn(1, max_text_len, hidden_dim))

        # image branch
        self.image_proj = nn.Linear(image_feat_dim, hidden_dim)

        # 0 = text, 1 = image
        self.modality_embedding = nn.Embedding(2, hidden_dim)

        # transformer
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # behavior branch
        self.behavior_mlp = nn.Sequential(
            nn.Linear(behavior_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )

        # fusion
        self.fusion_mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )

        # task heads
        self.primary_head = nn.Linear(hidden_dim, 1)  # harmful / benign logit
        self.report_head = nn.Linear(hidden_dim, 1)   # report-rate regression

    def forward(
        self,
        text_ids: torch.Tensor,
        text_mask: torch.Tensor,
        image_feats: torch.Tensor,
        behavior_feats: torch.Tensor,
    ):
        B, T = text_ids.shape
        _, I, _ = image_feats.shape

        # ----- text -----
        text_x = self.text_embedding(text_ids)                  # [B, T, text_embed_dim]
        text_x = self.text_proj(text_x)                         # [B, T, H]
        text_x = text_x + self.text_pos_embedding[:, :T, :]

        text_mod = torch.zeros(B, T, dtype=torch.long, device=text_ids.device)
        text_x = text_x + self.modality_embedding(text_mod)

        # ----- image -----
        image_x = self.image_proj(image_feats)                  # [B, I, H]
        image_mod = torch.ones(B, I, dtype=torch.long, device=text_ids.device)
        image_x = image_x + self.modality_embedding(image_mod)

        # ----- joint sequence -----
        joint_x = torch.cat([text_x, image_x], dim=1)           # [B, T+I, H]

        image_mask = torch.ones(B, I, dtype=text_mask.dtype, device=text_mask.device)
        joint_mask = torch.cat([text_mask, image_mask], dim=1)  # [B, T+I]
        padding_mask = joint_mask == 0                          # True = ignore

        encoded = self.transformer(joint_x, src_key_padding_mask=padding_mask)

        # masked mean pooling
        joint_mask_f = joint_mask.unsqueeze(-1).float()         # [B, T+I, 1]
        multimodal_repr = (encoded * joint_mask_f).sum(dim=1) / joint_mask_f.sum(dim=1).clamp(min=1.0)

        # behavior branch
        behavior_repr = self.behavior_mlp(behavior_feats)

        # fusion
        fused = torch.cat([multimodal_repr, behavior_repr], dim=-1)
        fused = self.fusion_mlp(fused)

        harmful_logit = self.primary_head(fused).squeeze(-1)    # [B]
        report_pred = self.report_head(fused).squeeze(-1)       # [B]

        return {
            "harmful_logit": harmful_logit,
            "report_pred": report_pred,
            "fused_repr": fused,
        }

In [7]:
# =========================================================
# 5. Loss + Metrics
# =========================================================
def compute_loss(outputs, harmful, report_rate, views, alpha=1.0, beta=0.5, c=5.0):
    harmful_logit = outputs["harmful_logit"]
    report_pred = outputs["report_pred"]

    # weighted BCE
    bce_per_sample = F.binary_cross_entropy_with_logits(
        harmful_logit,
        harmful,
        reduction="none",
    )
    weights = torch.clamp(torch.log1p(views), max=c)
    l_primary = (bce_per_sample * weights).mean()

    # regression loss
    l_reports = F.mse_loss(report_pred, report_rate)

    total_loss = alpha * l_primary + beta * l_reports
    return total_loss, l_primary, l_reports

In [8]:
@torch.no_grad()
def batch_metrics(outputs, harmful, report_rate):
    harmful_prob = torch.sigmoid(outputs["harmful_logit"])
    harmful_pred = (harmful_prob >= 0.5).float()
    acc = (harmful_pred == harmful).float().mean().item()

    mse = F.mse_loss(outputs["report_pred"], report_rate).item()
    return acc, mse

In [9]:
# =========================================================
# 6. Train / Eval loops
# =========================================================
def move_batch_to_device(batch, device):
    return {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}


def train_one_epoch(model, loader, optimizer, device, cfg: Config):
    model.train()

    total_loss = 0.0
    total_primary = 0.0
    total_reports = 0.0
    total_acc = 0.0
    total_mse = 0.0
    n_batches = 0

    for batch in loader:
        batch = move_batch_to_device(batch, device)

        optimizer.zero_grad()

        outputs = model(
            text_ids=batch["text_ids"],
            text_mask=batch["text_mask"],
            image_feats=batch["image_feats"],
            behavior_feats=batch["behavior_feats"],
        )

        loss, l_primary, l_reports = compute_loss(
            outputs=outputs,
            harmful=batch["harmful"],
            report_rate=batch["report_rate"],
            views=batch["views"],
            alpha=cfg.alpha,
            beta=cfg.beta,
            c=cfg.c,
        )

        loss.backward()
        optimizer.step()

        acc, mse = batch_metrics(outputs, batch["harmful"], batch["report_rate"])

        total_loss += loss.item()
        total_primary += l_primary.item()
        total_reports += l_reports.item()
        total_acc += acc
        total_mse += mse
        n_batches += 1

    return {
        "loss": total_loss / n_batches,
        "primary_loss": total_primary / n_batches,
        "report_loss": total_reports / n_batches,
        "acc": total_acc / n_batches,
        "report_mse": total_mse / n_batches,
    }

In [10]:
@torch.no_grad()
def evaluate(model, loader, device, cfg: Config):
    model.eval()

    total_loss = 0.0
    total_primary = 0.0
    total_reports = 0.0
    total_acc = 0.0
    total_mse = 0.0
    n_batches = 0

    for batch in loader:
        batch = move_batch_to_device(batch, device)

        outputs = model(
            text_ids=batch["text_ids"],
            text_mask=batch["text_mask"],
            image_feats=batch["image_feats"],
            behavior_feats=batch["behavior_feats"],
        )

        loss, l_primary, l_reports = compute_loss(
            outputs=outputs,
            harmful=batch["harmful"],
            report_rate=batch["report_rate"],
            views=batch["views"],
            alpha=cfg.alpha,
            beta=cfg.beta,
            c=cfg.c,
        )

        acc, mse = batch_metrics(outputs, batch["harmful"], batch["report_rate"])

        total_loss += loss.item()
        total_primary += l_primary.item()
        total_reports += l_reports.item()
        total_acc += acc
        total_mse += mse
        n_batches += 1

    return {
        "loss": total_loss / n_batches,
        "primary_loss": total_primary / n_batches,
        "report_loss": total_reports / n_batches,
        "acc": total_acc / n_batches,
        "report_mse": total_mse / n_batches,
    }

In [11]:
def print_metrics(stage, epoch, metrics):
    print(
        f"{stage:<5} Epoch {epoch:>2} | "
        f"loss={metrics['loss']:.4f} | "
        f"cls_loss={metrics['primary_loss']:.4f} | "
        f"rep_loss={metrics['report_loss']:.4f} | "
        f"acc={metrics['acc']:.4f} | "
        f"report_mse={metrics['report_mse']:.4f}"
    )

In [12]:
# =========================================================
# 7. QAT helpers
# =========================================================
def prepare_qat(model):
    if not HAS_TORCHAO:
        raise ImportError(
            "torchao is not installed. Install it first to use QAT."
        )

    qat_quantizer = Int8DynActInt4WeightQATQuantizer()
    model = qat_quantizer.prepare(model)
    return model, qat_quantizer

In [13]:
cfg = Config()
set_seed(cfg.seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
print("TorchAO available:", HAS_TORCHAO)

Using device: cpu
TorchAO available: False


In [14]:
# ----- datasets -----
train_dataset = MultimodalModerationDataset(
    num_samples=cfg.train_size,
    vocab_size=cfg.vocab_size,
    max_text_len=cfg.max_text_len,
    num_image_tokens=cfg.num_image_tokens,
    image_feat_dim=cfg.image_feat_dim,
    behavior_dim=cfg.behavior_dim,
)

test_dataset = MultimodalModerationDataset(
    num_samples=cfg.test_size,
    vocab_size=cfg.vocab_size,
    max_text_len=cfg.max_text_len,
    num_image_tokens=cfg.num_image_tokens,
    image_feat_dim=cfg.image_feat_dim,
    behavior_dim=cfg.behavior_dim,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.batch_size,
    shuffle=True,
    collate_fn=collate_fn,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=cfg.batch_size,
    shuffle=False,
    collate_fn=collate_fn,
)

In [16]:
# ----- model -----
model = MultimodalClassifier(
    vocab_size=cfg.vocab_size,
    text_embed_dim=cfg.text_embed_dim,
    image_feat_dim=cfg.image_feat_dim,
    behavior_dim=cfg.behavior_dim,
    hidden_dim=cfg.hidden_dim,
    num_heads=cfg.num_heads,
    num_layers=cfg.num_layers,
    ff_dim=cfg.ff_dim,
    max_text_len=cfg.max_text_len,
    dropout=cfg.dropout,
).to(device)

# ----- normal FP training -----
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=cfg.lr,
    weight_decay=cfg.weight_decay,
)

In [17]:
print("\n=== FP Training ===")
for epoch in range(1, cfg.epochs + 1):
    train_metrics = train_one_epoch(model, train_loader, optimizer, device, cfg)
    test_metrics = evaluate(model, test_loader, device, cfg)

    print_metrics("Train", epoch, train_metrics)
    print_metrics("Test", epoch, test_metrics)


=== FP Training ===
Train Epoch  1 | loss=3.5203 | cls_loss=3.4685 | rep_loss=0.1035 | acc=0.5127 | report_mse=0.1035
Test  Epoch  1 | loss=3.5284 | cls_loss=3.4876 | rep_loss=0.0817 | acc=0.4911 | report_mse=0.0817
Train Epoch  2 | loss=3.5081 | cls_loss=3.4633 | rep_loss=0.0897 | acc=0.5068 | report_mse=0.0897
Test  Epoch  2 | loss=3.5335 | cls_loss=3.4859 | rep_loss=0.0953 | acc=0.4911 | report_mse=0.0953
Train Epoch  3 | loss=3.5087 | cls_loss=3.4648 | rep_loss=0.0878 | acc=0.5156 | report_mse=0.0878
Test  Epoch  3 | loss=3.5048 | cls_loss=3.4633 | rep_loss=0.0830 | acc=0.4821 | report_mse=0.0830
Train Epoch  4 | loss=3.4795 | cls_loss=3.4371 | rep_loss=0.0848 | acc=0.5410 | report_mse=0.0848
Test  Epoch  4 | loss=3.5246 | cls_loss=3.4819 | rep_loss=0.0854 | acc=0.5089 | report_mse=0.0854
Train Epoch  5 | loss=3.4480 | cls_loss=3.4023 | rep_loss=0.0914 | acc=0.5645 | report_mse=0.0914
Test  Epoch  5 | loss=3.5904 | cls_loss=3.5469 | rep_loss=0.0868 | acc=0.4911 | report_mse=0.0868

In [18]:
# ----- optional QAT fine-tuning -----
qat_quantizer = None
if cfg.use_qat:
    if not HAS_TORCHAO:
        print("\nQAT requested, but torchao is not installed. Skipping QAT.")
    else:
        print("\n=== Preparing QAT ===")
        model, qat_quantizer = prepare_qat(model)

        # re-create optimizer after prepare()
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=cfg.lr * 0.1,  # smaller lr for QAT fine-tuning
            weight_decay=cfg.weight_decay,
        )

        print("=== QAT Fine-tuning ===")
        for epoch in range(1, cfg.qat_finetune_epochs + 1):
            train_metrics = train_one_epoch(model, train_loader, optimizer, device, cfg)
            test_metrics = evaluate(model, test_loader, device, cfg)

            print_metrics("QAT-T", epoch, train_metrics)
            print_metrics("QAT-V", epoch, test_metrics)

        print("\n=== Converting to quantized model ===")
        model = qat_quantizer.convert(model)


QAT requested, but torchao is not installed. Skipping QAT.


In [19]:
# ----- inference example -----
print("\n=== Inference Example ===")
model.eval()

batch = next(iter(test_loader))
batch = move_batch_to_device(batch, device)

with torch.no_grad():
    outputs = model(
        text_ids=batch["text_ids"],
        text_mask=batch["text_mask"],
        image_feats=batch["image_feats"],
        behavior_feats=batch["behavior_feats"],
    )

harmful_prob = torch.sigmoid(outputs["harmful_logit"])
report_pred = outputs["report_pred"]

print("harmful_prob shape:", harmful_prob.shape)
print("report_pred shape :", report_pred.shape)
print("first 5 harmful probs:", harmful_prob[:5].detach().cpu())
print("first 5 report preds :", report_pred[:5].detach().cpu())


=== Inference Example ===
harmful_prob shape: torch.Size([32])
report_pred shape : torch.Size([32])
first 5 harmful probs: tensor([0.3791, 0.5893, 0.3992, 0.6476, 0.5034])
first 5 report preds : tensor([0.5212, 0.5235, 0.5101, 0.5985, 0.5071])
